# Computing each term of the matrix-free $W u$

For the pairwise-epistasis GRM
$$W = \frac{1}{p}\sum_{a<b} h_{ab}h_{ab}^\top,\qquad h_{ab}=\frac{d_{ab}-E_{ab}\mathbf 1}{\sqrt{V_{ab}}},\qquad d_{ab}=Z_{\cdot a}\circ Z_{\cdot b},\qquad p=\binom{m}{2},$$
the apply $W u$ expands into **four terms**, written with the weight matrices $S,R,T$, the genotype $Z$, and $u$ only:
$$W u = \frac1p\Big[\underbrace{\tfrac12\big(Z\circ(Z(S\circ M))\big)\mathbf 1}_{\texttt{term\_dd}}\;-\;\underbrace{\tfrac{s_u}{2}\big(Z\circ(ZR)\big)\mathbf 1}_{\texttt{term\_mu}}\;-\;\underbrace{\tfrac12\big(\mathbf 1^\top(R\circ M)\mathbf 1\big)\mathbf 1}_{\texttt{term\_dm}}\;+\;\underbrace{\tfrac{s_u}{2}\big(\mathbf 1^\top T\mathbf 1\big)\mathbf 1}_{\texttt{term\_TT}}\Big]$$
where
$$M = Z^\top\operatorname{diag}(u)\,Z\in\mathbb R^{m\times m},\qquad s_u=\mathbf 1^\top u=\sum_i u_i,$$
$A\mathbf 1$ is the **row-sum** of $A$ (a vector in $\mathbb R^n$) and $\mathbf 1^\top A\mathbf 1$ is the **sum of all entries** of $A$ (a scalar).

This notebook computes each term separately and checks that they sum to the explicit $W u$.

In [1]:
import numpy as np
np.set_printoptions(precision=4, suppress=True)

## 1. Inputs: genotype $Z$, weight matrices $S,R,T$, and a vector $u$

In [2]:
def standardize_cols(M, eps=1e-12):
    """Column-standardize to mean 0, variance 1 (ddof=0)."""
    M = np.asarray(M, dtype=float)
    mu = M.mean(axis=0)
    sd = M.std(axis=0)
    sd = np.where(sd < eps, 1.0, sd)
    return (M - mu) / sd


def weight_matrices(Z):
    """m-by-m (S, R, T) from Z only.
        E  = (Z^T Z)/n
        Vp = ((Z.Z)^T (Z.Z))/n - E^2
        S = 1/Vp,  R = E/Vp,  T = E^2/Vp   (diagonals zeroed)
    """
    n = Z.shape[0]
    E = (Z.T @ Z) / n
    D = Z * Z                      # element-wise square (Z .circ Z)
    Vp = (D.T @ D) / n - E * E
    S = 1.0 / Vp
    R = E / Vp
    T = (E * E) / Vp
    np.fill_diagonal(S, 0.0)
    np.fill_diagonal(R, 0.0)
    np.fill_diagonal(T, 0.0)
    return S, R, T


# --- build a small, non-degenerate example ---
rng = np.random.default_rng(0)
n, m = 200, 30
Z = standardize_cols(rng.standard_normal((n, m)))   # column-standardized genotype
u = rng.standard_normal(n)                          # the vector W is applied to

S, R, T = weight_matrices(Z)
p = m * (m - 1) // 2
print(f"n={n}, m={m}, p=C(m,2)={p}")
print(f"Z {Z.shape},  u {u.shape},  S/R/T {S.shape}")

n=200, m=30, p=C(m,2)=435
Z (200, 30),  u (200,),  S/R/T (30, 30)


## 2. The two shared pieces: $s_u$ and $M = Z^\top\operatorname{diag}(u)Z$

In [3]:
s_u = u.sum()                       # 1^T u
M = Z.T @ (u[:, None] * Z)          # Z^T diag(u) Z,  shape (m, m)

print(f"s_u = 1^T u = {s_u:.6f}")
print(f"M = Z^T diag(u) Z  ->  {M.shape}")
# sanity: M_ab = sum_i u_i Z_ia Z_ib
a, b = 3, 7
print(f"check M[{a},{b}]: {M[a, b]:.6f}  vs  {np.sum(u * Z[:, a] * Z[:, b]):.6f}")

s_u = 1^T u = -7.332555
M = Z^T diag(u) Z  ->  (30, 30)
check M[3,7]: -6.493396  vs  -6.493396


## 3. Term 1 &mdash; `term_dd` $=\tfrac12\big(Z\circ(Z(S\circ M))\big)\mathbf 1$

The $d_{ab}(d_{ab}^\top u)$ part, weighted by $S=1/V_p$. Per individual $i$ it is the quadratic form $\tfrac12\,z_i^\top(S\circ M)z_i$; stacked over $i$ this is the row-sum of $Z\circ(Z(S\circ M))$. It is a **vector** in $\mathbb R^n$.

In [4]:
term_dd = 0.5 * np.sum(Z * (Z @ (S * M)), axis=1)   # (n,)
print(f"term_dd: shape {term_dd.shape},  first 5 = {term_dd[:5]}")

term_dd: shape (200,),  first 5 = [-108.199    34.9683  203.7779  160.2459  958.5754]


## 4. Term 2 &mdash; `term_mu` $=\tfrac{s_u}{2}\big(Z\circ(ZR)\big)\mathbf 1$

The $-E_{ab}(\mathbf 1^\top u)\,d_{ab}$ part, weighted by $R=E/V_p$. Per individual $i$ it is $\tfrac{s_u}{2}\,z_i^\top R\,z_i$ &mdash; depends on $u$ **only through** $s_u$, not $M$. Also a **vector** in $\mathbb R^n$. (Enters $W u$ with a minus sign.)

In [5]:
quad_R = 0.5 * np.sum(Z * (Z @ R), axis=1)          # 0.5 * z_i^T R z_i,  (n,)
term_mu = s_u * quad_R                              # (n,)
print(f"term_mu: shape {term_mu.shape},  first 5 = {term_mu[:5]}")

term_mu: shape (200,),  first 5 = [ -2.4517  -8.0353 -36.2568 -13.1632  -6.6727]


## 5. Term 3 &mdash; `term_dm` $=\tfrac12\big(\mathbf 1^\top(R\circ M)\mathbf 1\big)\mathbf 1$

The $-E_{ab}(d_{ab}^\top u)\,\mathbf 1$ part, weighted by $R$. It is a **scalar** (identical for every individual): the total sum of $R\circ M$. It becomes the constant vector $\text{term\_dm}\cdot\mathbf 1$. (Enters $W u$ with a minus sign.)

In [6]:
term_dm = 0.5 * np.sum(R * M)                       # scalar
print(f"term_dm (scalar) = {term_dm:.6f}")

term_dm (scalar) = -20.226968


## 6. Term 4 &mdash; `term_TT` $=\tfrac{s_u}{2}\big(\mathbf 1^\top T\mathbf 1\big)\mathbf 1$

The $+E_{ab}^2(\mathbf 1^\top u)\,\mathbf 1$ part, weighted by $T=E^2/V_p$. Also a **scalar** (identical for every individual): $\tfrac{s_u}{2}$ times the total sum of $T$. Becomes the constant vector $\text{term\_TT}\cdot\mathbf 1$.

In [7]:
sum_T = 0.5 * np.sum(T)                             # scalar
term_TT = s_u * sum_T                               # scalar
print(f"sum_T = 0.5 * sum(T) = {sum_T:.6f}")
print(f"term_TT (scalar)    = {term_TT:.6f}")

sum_T = 0.5 * sum(T) = 2.369494
term_TT (scalar)    = -17.374443


## 7. Assemble  $W u = \tfrac1p(\texttt{term\_dd} - \texttt{term\_mu} - \texttt{term\_dm} + \texttt{term\_TT})$

In [8]:
Wu_implicit = (term_dd - term_mu - term_dm + term_TT) / p   # (n,)  (scalars broadcast)
print(f"Wu_implicit: shape {Wu_implicit.shape},  first 5 = {Wu_implicit[:5]}")

# per-term contribution to the norm of Wu (each divided by p, scalars as constant vectors)
ones = np.ones(n)
contrib = {
    'term_dd':  term_dd / p,
    '-term_mu': -term_mu / p,
    '-term_dm': -term_dm * ones / p,
    '+term_TT': term_TT * ones / p,
}
print("\nper-term L2 norm of contribution to Wu:")
for name, v in contrib.items():
    print(f"  {name:>9}:  ||.||_2 = {np.linalg.norm(v):.6e}")

Wu_implicit: shape (200,),  first 5 = [-0.2365  0.1054  0.5584  0.4052  2.2255]

per-term L2 norm of contribution to Wu:
    term_dd:  ||.||_2 = 1.645978e+01
   -term_mu:  ||.||_2 = 7.239894e-01
   -term_dm:  ||.||_2 = 6.575920e-01
   +term_TT:  ||.||_2 = 5.648545e-01


## 8. Check against the explicit dense $W u$

Build $W=\tfrac1p\sum_{a<b}h_{ab}h_{ab}^\top$ densely and compare $W u$.

In [9]:
def build_W_explicit(Z):
    """Dense W = (1/p) sum_{a<b} h_ab h_ab',  h_ab = std(Z_a . Z_b)."""
    n, m = Z.shape
    ia, ib = np.triu_indices(m, k=1)
    H = Z[:, ia] * Z[:, ib]                 # raw pair products, (n, p)
    H = (H - H.mean(axis=0)) / H.std(axis=0, ddof=0)   # standardize each column
    return (H @ H.T) / H.shape[1]


W = build_W_explicit(Z)
Wu_explicit = W @ u

abs_err = np.max(np.abs(Wu_implicit - Wu_explicit))
rel_err = abs_err / np.max(np.abs(Wu_explicit))
print(f"max abs error = {abs_err:.3e}")
print(f"max rel error = {rel_err:.3e}")
assert rel_err < 1e-8, "terms do NOT reassemble the explicit W u"
print("\nOK: term_dd - term_mu - term_dm + term_TT  (/p)  ==  explicit W u")

max abs error = 3.553e-15
max rel error = 9.402e-16

OK: term_dd - term_mu - term_dm + term_TT  (/p)  ==  explicit W u


## 9. Summary table of the four terms

In [10]:
print(f"{'term':>10} {'sign':>5} {'type':>8}   formula")
print('-' * 72)
rows = [
    ('term_dd', '+', 'vector', '0.5 * rowsum( Z .* (Z @ (S.*M)) )'),
    ('term_mu', '-', 'vector', '(s_u/2) * rowsum( Z .* (Z @ R) )'),
    ('term_dm', '-', 'scalar', '0.5 * sum( R .* M )'),
    ('term_TT', '+', 'scalar', '(s_u/2) * sum( T )'),
]
for name, sign, typ, f in rows:
    print(f"{name:>10} {sign:>5} {typ:>8}   {f}")
print('-' * 72)
print("Wu = ( term_dd - term_mu - term_dm + term_TT ) / p     with p = C(m,2)")
print("M = Z^T diag(u) Z,   s_u = sum(u),   .* = element-wise,   rowsum = sum over columns")

      term  sign     type   formula
------------------------------------------------------------------------
   term_dd     +   vector   0.5 * rowsum( Z .* (Z @ (S.*M)) )
   term_mu     -   vector   (s_u/2) * rowsum( Z .* (Z @ R) )
   term_dm     -   scalar   0.5 * sum( R .* M )
   term_TT     +   scalar   (s_u/2) * sum( T )
------------------------------------------------------------------------
Wu = ( term_dd - term_mu - term_dm + term_TT ) / p     with p = C(m,2)
M = Z^T diag(u) Z,   s_u = sum(u),   .* = element-wise,   rowsum = sum over columns


---

# 10. Reducing the cost: the identities behind the precomputation

Sections 1-9 cost $\approx 3nm^2$ per apply, in three products: $M=Z^\top\operatorname{diag}(u)Z$, $Z(S\circ M)$, and $ZR$. This section verifies the two identities that remove one of them (`Wu_complexity.typ`, *Precompute the elements independent of $u$*).

**Naming.** This notebook's `S` $=1/V_p$ is the note's $V$. The note's $V\circ M$ is the expression `S * M` here, written `Sn` below to avoid the clash.

## 10.1 The $\operatorname{dg}$ identity

Write $\operatorname{dg}(A)$ for the vector of diagonal entries of a square $A$ (the *opposite* of $\operatorname{diag}(v)$, which builds a matrix from a vector). Then

$$\big(Z\circ(Z S_n)\big)\mathbf 1 \;=\; \operatorname{dg}(Z S_n Z^\top),\qquad \text{row } i = z_i^\top S_n z_i,$$

and if $S_n$ is symmetric with zero diagonal (it is, since $V$ and $M$ both are), the quadratic form collapses onto the $a<b$ pair loop:

$$z_i^\top S_n z_i \;=\; \sum_{a,b} (S_n)_{ab}Z_{ia}Z_{ib} \;=\; 2\sum_{a<b}(S_n)_{ab}Z_{ia}Z_{ib}.$$

Three expressions, one vector - but very different cost:

| Level | Expression | Time | Space |
|---|---|---|---|
| 1 | $\operatorname{dg}(ZS_nZ^\top)$, formed literally | $O(n^2m)$ | $O(n^2)$ &nbsp;&#10007; |
| 2 | $(Z\circ(ZS_n))\mathbf 1$ | $nm^2$ | $O(nm)$ |
| 3 | $2\sum_{a<b}(S_n)_{ab}Z_{ia}Z_{ib}$ | $nm^2/2$ | $O(1)$ |

Level 1 is the *meaning*, and is exactly the $O(n^2)$ blow-up the matrix-free apply exists to avoid. **Level 2 is what the apply uses** (section 3, and the first bracket of the final formula). Level 3 is verified here for completeness but is not used - see Appendix A.

In [11]:
Sn = S * M                      # the note's  S = V .* M   (symmetric, zero diagonal)
print(f"Sn symmetric?      {np.allclose(Sn, Sn.T)}")
print(f"Sn zero diagonal?  {np.allclose(np.diag(Sn), 0)}")

# level 1 -- the literal meaning: form the n-by-n matrix, keep its diagonal
lvl1 = np.diag(Z @ Sn @ Z.T)

# level 2 -- row-sums, never forming an n-by-n object  (this is section 3)
lvl2 = np.sum(Z * (Z @ Sn), axis=1)

# level 3 -- walk the a<b pairs only, using symmetry
ia, ib = np.triu_indices(m, k=1)
lvl3 = 2.0 * (Z[:, ia] * Z[:, ib]) @ Sn[ia, ib]

print(f"\nlevel 1 first 3 = {lvl1[:3]}")
print(f"level 2 first 3 = {lvl2[:3]}")
print(f"level 3 first 3 = {lvl3[:3]}")
print(f"\nmax |lvl1 - lvl2| = {np.max(np.abs(lvl1 - lvl2)):.3e}")
print(f"max |lvl2 - lvl3| = {np.max(np.abs(lvl2 - lvl3)):.3e}")

# and each reproduces term_dd of section 3 (which carries the 0.5)
print(f"max |0.5*lvl3 - term_dd| = {np.max(np.abs(0.5 * lvl3 - term_dd)):.3e}")
assert np.allclose(lvl1, lvl2) and np.allclose(lvl2, lvl3)
print("\nOK: dg(Z Sn Z^T) = (Z .* (Z Sn)) 1 = 2 * sum_{a<b} Sn_ab Z_ia Z_ib")

Sn symmetric?      True
Sn zero diagonal?  True

level 1 first 3 = [-216.3981   69.9366  407.5557]
level 2 first 3 = [-216.3981   69.9366  407.5557]
level 3 first 3 = [-216.3981   69.9366  407.5557]

max |lvl1 - lvl2| = 4.547e-13
max |lvl2 - lvl3| = 3.638e-12
max |0.5*lvl3 - term_dd| = 1.819e-12

OK: dg(Z Sn Z^T) = (Z .* (Z Sn)) 1 = 2 * sum_{a<b} Sn_ab Z_ia Z_ib


## 10.2 `term_dm` needs no $M$

The same $\operatorname{dg}$ pattern applied to $R$ gives a vector that depends on $Z$ alone:

$$v_R \;=\; \big(Z\circ(ZR)\big)\mathbf 1 \;=\; \operatorname{dg}(ZRZ^\top)\in\mathbb R^n .$$

`term_mu` already uses it (it is $2\times$`quad_R`). The point is that `term_dm` can use it too:

$$\mathbf 1^\top(R\circ M)\mathbf 1 \;=\; \langle R,M\rangle \;=\; \operatorname{tr}(RM) \;=\; \operatorname{tr}\!\big(R\,Z^\top\operatorname{diag}(u)Z\big) \;=\; \operatorname{tr}\!\big(\operatorname{diag}(u)\,ZRZ^\top\big) \;=\; u^\top v_R,$$

using $R=R^\top$ in the second step and cyclicity of the trace in the fourth. So `term_dm` is an $O(n)$ dot product against a cached vector, and $ZR$ - one of the three $nm^2$ products - leaves the apply entirely.

In [13]:
v_R = np.sum(Z * (Z @ R), axis=1)        # = dg(Z R Z^T),  depends on Z only
s_T = np.sum(T)                          # = 1^T T 1,      depends on Z only

print(f"v_R = dg(Z R Z^T)?        {np.max(np.abs(v_R - np.diag(Z @ R @ Z.T))):.3e}")
print(f"v_R = 2 * quad_R?         {np.max(np.abs(v_R - 2 * quad_R)):.3e}")

# the chain of section 10.2, one equality at a time
lhs   = np.sum(R * M)                    # 1^T (R .* M) 1  =  <R, M>
step1 = np.trace(R @ M)                  # = tr(R M)
step2 = np.trace(np.diag(u) @ Z @ R @ Z.T)   # = tr(diag(u) Z R Z^T)   [cyclic]
rhs   = u @ v_R                          # = u^T v_R

print(f"\n1'(R.*M)1              = {lhs: .6f}")
print(f"tr(R M)                = {step1: .6f}")
print(f"tr(diag(u) Z R Z^T)    = {step2: .6f}")
print(f"u^T v_R                = {rhs: .6f}")

# and this reproduces term_dm of section 5 (which carries the 0.5)
print(f"\nmax |0.5*u^T v_R - term_dm| = {abs(0.5 * rhs - term_dm):.3e}")
assert np.allclose([step1, step2, rhs], lhs)
print("\nOK: term_dm computable from the cached v_R -- no M, no Z@R, O(n)")

v_R = dg(Z R Z^T)?        5.329e-15
v_R = 2 * quad_R?         0.000e+00

1'(R.*M)1              = -40.453935
tr(R M)                = -40.453935
tr(diag(u) Z R Z^T)    = -40.453935
u^T v_R                = -40.453935

max |0.5*u^T v_R - term_dm| = 3.553e-15

OK: term_dm computable from the cached v_R -- no M, no Z@R, O(n)


# 11. The final formula

With $v_R$ and $s_T$ precomputed from $Z$ alone, three of the four terms collapse into scalars times cached vectors, and the four-term expression of §7 becomes

$$\boxed{\;W u \;=\; \frac{1}{2p}\Big[\;\operatorname{dg}(Z S_n Z^\top)\;-\;s_u\,v_R\;+\;\big(s_u s_T - u^\top v_R\big)\mathbf 1\;\Big],\qquad S_n = V\circ\big(Z^\top\operatorname{diag}(u)Z\big).\;}$$

Only the first term costs real work. The $\tfrac12$'s of §§3–6 have been folded into the single $1/(2p)$ out front — that factor of 2 is exactly the ordered/unordered pair double-count.

**Setup, once per gene:** $V,R,T$, then $v_R=\operatorname{dg}(ZRZ^\top)$ and $s_T=\mathbf 1^\top T\mathbf 1$. Afterwards **$R$, $T$ and $ZR$ can all be discarded** — only $Z$, $V$, $v_R$, $s_T$ stay live.

**Per apply:** one $u$-dependent object, $M=Z^\top\operatorname{diag}(u)Z$.

In [13]:
def setup_once(Z):
    """Everything that depends on Z alone. R, T and Z@R are dropped afterwards."""
    V, R, T = weight_matrices(Z)                 # note's V is this file's S
    v_R = np.sum(Z * (Z @ R), axis=1)            # dg(Z R Z^T),  (n,)
    s_T = np.sum(T)                              # 1^T T 1,      scalar
    return V, v_R, s_T                           # R, T no longer needed


def apply_W_final(Z, u, V, v_R, s_T):
    """The boxed formula. Only Z, V, v_R, s_T -- no R, no T."""
    n, m = Z.shape
    p = m * (m - 1) // 2
    s_u = u.sum()
    M = Z.T @ (u[:, None] * Z)                   # the one u-dependent object
    t1 = np.sum(Z * (Z @ (V * M)), axis=1)       # dg(Z Sn Z^T)
    return (t1 - s_u * v_R + (s_u * s_T - u @ v_R)) / (2 * p)


V_, v_R_, s_T_ = setup_once(Z)
Wu_final = apply_W_final(Z, u, V_, v_R_, s_T_)

print(f"Wu_final    first 5 = {Wu_final[:5]}")
print(f"Wu_implicit first 5 = {Wu_implicit[:5]}")
print(f"Wu_explicit first 5 = {Wu_explicit[:5]}")
print(f"\nmax |Wu_final - Wu_implicit| = {np.max(np.abs(Wu_final - Wu_implicit)):.3e}")
print(f"max |Wu_final - Wu_explicit| = {np.max(np.abs(Wu_final - Wu_explicit)):.3e}")

assert np.allclose(Wu_final, Wu_explicit)
print("\nOK: boxed formula == 4-term expansion == dense W u")

Wu_final    first 5 = [-0.2365  0.1054  0.5584  0.4052  2.2255]
Wu_implicit first 5 = [-0.2365  0.1054  0.5584  0.4052  2.2255]
Wu_explicit first 5 = [-0.2365  0.1054  0.5584  0.4052  2.2255]

max |Wu_final - Wu_implicit| = 4.441e-16
max |Wu_final - Wu_explicit| = 3.553e-15

OK: boxed formula == 4-term expansion == dense W u
